# 04 - Metadatos, linaje y privacidad en Python

## Objetivo y alcance

Relacionar un resultado del ETL con su significado, procedencia y condiciones de acceso. Todo funciona localmente con pandas y modulos `json`, `hashlib`, `hmac` y `secrets` de Python. No se integra un catalogo cloud ni un sistema real de identidad. Los datos son ficticios y los emails usan dominios reservados `.invalid`; nunca reemplazarlos por datos personales reales para practicar.

## Paso 1 - Del glosario al catalogo y al linaje

El **glosario** define terminos de negocio; el **diccionario** describe columnas y tipos; el **catalogo** identifica el activo, Owner, Steward, finalidad, clasificacion y condiciones de uso. Deben mantenerse conectados. Un nombre tecnico no sustituye una definicion: `monto_centavos` es el importe en la moneda original y no un total convertido a una moneda unica.

`metadatos()` retorna un catalogo JSON con grano, roles y mapeos por campo. La retencion queda pendiente de aprobacion: declarar que falta una decision es mas honesto que inventar un plazo universal. En produccion este estado debe impedir tratamientos que requieran esa aprobacion.

`linaje()` identifica entidad fuente, actividad, agente, contrato y destino, y asocia los campos con origen y transformacion. Se inspira en los conceptos de W3C PROV, pero no es una serializacion conforme a PROV. La huella SHA-256 permite comparar contenido representado por la misma funcion; no prueba por si sola quien lo produjo, su exactitud ni que no haya sido manipulado antes de calcularla.

Se generaran documentos JSON en memoria para inspeccionarlos. No se escribe informacion personal ni secretos en archivos.

### Organizacion del codigo

Las funciones de catalogo, linaje, huella, tokenizacion y vistas por rol se definen en la siguiente celda de este notebook. Son parte de esta explicacion, no del nucleo comun a los cinco ejemplos. Leer sus definiciones permite verificar que campos se muestran, como se calcula el token y que limitaciones tiene la simulacion.

Solo el conjunto ficticio, la normalizacion, la evaluacion y la version del contrato provienen de `gobierno_demo.py`. Las llamadas a `metadatos`, `linaje`, `token_persona` y `vista_por_rol` usan implementaciones locales visibles.

In [ ]:
import hashlib
import hmac

import pandas as pd


def huella(datos):
    contenido = datos.to_json(orient='split', date_format='iso', index=False)
    return hashlib.sha256(contenido.encode('utf-8')).hexdigest()


def token_persona(valor, clave):
    if len(clave) < 32:
        raise ValueError('Usar una clave aleatoria de al menos 32 bytes para la demo')
    if pd.isna(valor) or not str(valor).strip():
        return None
    mensaje = f'demo:email:{str(valor).strip().lower()}'.encode('utf-8')
    return hmac.new(clave, mensaje, hashlib.sha256).hexdigest()


def vista_por_rol(datos, rol, clave):
    columnas = ['id_venta', 'id_producto', 'fecha_venta', 'monto_centavos', 'moneda']
    if rol == 'analista':
        return datos[columnas].copy()
    if rol == 'steward':
        resultado = datos[columnas].copy()
        resultado['email_token'] = datos['email'].map(lambda valor: token_persona(valor, clave))
        return resultado
    raise PermissionError('Rol sin vista autorizada en la simulacion')


def metadatos():
    return {
        'activo': 'silver.ventas', 'version': '1.0.0', 'grano': 'una fila por id_venta',
        'owner': 'Gerencia Comercial', 'steward': 'Steward Comercial',
        'finalidad': 'analitica de ventas', 'clasificacion': 'confidencial',
        'retencion': 'por definir y aprobar antes de produccion',
        'columnas': [
            {'nombre': 'id_venta', 'tipo': 'INTEGER', 'nullable': False, 'termino': 'Identificador de venta', 'origen': 'origen.id_venta', 'transformacion': 'entero positivo', 'reglas': ['DQ_ID', 'DQ_UNICIDAD']},
            {'nombre': 'id_producto', 'tipo': 'INTEGER', 'nullable': False, 'termino': 'Producto de la venta', 'origen': 'origen.id_producto', 'transformacion': 'lookup en productos', 'reglas': ['DQ_FK_PRODUCTO']},
            {'nombre': 'fecha_venta', 'tipo': 'DATE', 'nullable': False, 'termino': 'Fecha del evento', 'origen': 'origen.fecha_venta', 'transformacion': 'parseo ISO', 'reglas': ['DQ_FECHA']},
            {'nombre': 'monto_centavos', 'tipo': 'INTEGER', 'nullable': False, 'termino': 'Importe de linea en moneda original', 'origen': 'origen.monto', 'transformacion': 'Decimal por 100, sin conversion de moneda', 'reglas': ['DQ_MONTO']},
            {'nombre': 'moneda', 'tipo': 'TEXT', 'nullable': False, 'termino': 'Moneda original', 'origen': 'origen.moneda', 'transformacion': 'strip y upper', 'reglas': ['DQ_MONEDA']},
        ],
    }


def linaje(datos):
    return {
        'entidad_fuente': 'ventas_ficticias', 'huella': huella(datos),
        'actividad': 'normalizar_validar_publicar', 'agente': 'servicio_etl_demo',
        'contrato': demo.VERSION_CONTRATO, 'entidad_destino': 'silver.ventas',
        'campos': metadatos()['columnas'],
        'limite': 'modelo didactico inspirado en PROV, no serializacion PROV conforme',
    }


def json_seguro(objeto):
    return json.dumps(objeto, ensure_ascii=True, indent=2, allow_nan=False)

In [ ]:
import json
import secrets
import sys
from pathlib import Path

candidatos = [Path.cwd(), Path.cwd() / 'notebook' / 'unidad_III']
carpeta = next((ruta for ruta in candidatos if (ruta / 'gobierno_demo.py').exists()), None)
if carpeta is None:
    raise RuntimeError('Ejecutar desde la raiz del repositorio o notebook/unidad_III')
sys.path.insert(0, str(carpeta.resolve()))
import gobierno_demo as demo

crudos, productos, referencia = demo.crear_datos()
datos = demo.normalizar(crudos)
aceptadas, rechazadas, metricas, detalle = demo.evaluar(datos, productos, referencia)
catalogo = metadatos()
glosario = {
    'Venta': 'Evento identificado por id_venta, no una persona ni un cliente.',
    'Importe de linea': 'Valor positivo expresado en la moneda original de la venta.',
    'Fila apta': 'Sin fallos de cuarentena; puede conservar advertencias.',
}
procedencia = linaje(crudos)
print(json_seguro(glosario))
print(json_seguro(catalogo))
print(json_seguro(procedencia))
assert json.loads(json_seguro(catalogo))['grano'] == 'una fila por id_venta'
assert len(procedencia['campos']) == 5
assert huella(crudos) == huella(crudos.copy(deep=True))
alterados = crudos.copy(deep=True)
alterados.loc[0, 'monto'] = '1501.00'
assert huella(crudos) != huella(alterados)

## Paso 2 - Minimizacion y seudonimizacion

La vista del analista solo incluye campos necesarios para analitica; no contiene email ni token. La vista del Steward incorpora un token HMAC para vincular registros sin mostrar la direccion. Un rol no definido debe recibir `PermissionError`.

Esto es una **simulacion de politica en una funcion**, no autenticacion ni RBAC real. Quien tiene el dataframe crudo o puede ejecutar codigo en este proceso puede leer los datos. En produccion se requieren identidades verificadas, permisos en el almacenamiento, restricciones del servicio y pruebas con cuentas separadas.

HMAC usa un secreto aleatorio de 32 bytes. La misma direccion normalizada y la misma clave producen el mismo token; otra clave produce otro token. No se imprime ni se guarda la clave. El token no es cifrado reversible ni anonimiza irreversiblemente: sigue siendo un identificador vinculable y debe protegerse. Sin secreto, un hash de un email conocido puede atacarse por enumeracion.

La clave efimera cambia al reiniciar el notebook. Eso impide vinculacion estable entre sesiones deliberadamente. Una clave persistente exige gestion, rotacion, versionado y permisos apropiados. Ni la minimizacion ni HMAC reemplazan base legal, finalidad, plazo de retencion o analisis de riesgos.

In [ ]:
clave = secrets.token_bytes(32)
vista_analista = vista_por_rol(aceptadas, 'analista', clave)
vista_steward = vista_por_rol(aceptadas, 'steward', clave)
display(vista_analista)
display(vista_steward)
assert 'email' not in vista_analista.columns
assert 'email_token' not in vista_analista.columns
assert 'email' not in vista_steward.columns
assert 'email_token' in vista_steward.columns

acceso_denegado = False
try:
    vista_por_rol(aceptadas, 'visitante', clave)
except PermissionError:
    acceso_denegado = True
assert acceso_denegado

primer_token = token_persona('uno@demo.invalid', clave)
assert primer_token == token_persona(' UNO@DEMO.INVALID ', clave)
assert primer_token != token_persona('uno@demo.invalid', secrets.token_bytes(32))
assert token_persona(None, clave) is None
assert len(primer_token) == 64
del clave
print('Minimizacion, rechazo de rol y tokenizacion probados. No se imprime la clave.')

## Interpretacion y ejercicios

Se debe observar un catalogo con 5 campos de consumo, linaje por campo, una vista analitica sin datos de contacto y una vista del Steward con tokens. El acceso al rol visitante falla de forma esperada y se captura. No se imprime la clave de HMAC.

1. Agregar al catalogo propietario de la fuente, frecuencia, contrato, fecha de revision y criterio de retencion. Distinguir valor propuesto de aprobado.
2. Vincular una corrida real de prueba con su `id_lote` y los metadatos. La huella de un dataset no sustituye el identificador del intento.
3. Escribir una politica de finalidad para la vista del Steward y un procedimiento de solicitud, autorizacion, revocacion y revision de acceso.
4. Proponer como rotar claves sin perder toda vinculacion historica ni exponer el secreto. No implementar un repositorio de claves en el propio notebook.

**Limites:** no hay cifrado de disco, IAM, RBAC, borrado verificable ni proteccion frente a quien controla el proceso Python. `del clave` elimina una referencia de la sesion, no garantiza borrado seguro de memoria. No guardar outputs que contengan datos personales reales. En el manual se explican las medidas institucionales que faltan para produccion.